# Notebook 09 – Frontend Implementation with Next.js

## 1. Overview
This notebook implements the frontend for the **Advance AI Research Assistant** using Next.js, JavaScript, and CSS. The frontend communicates with the FastAPI backend to support PDF uploads and scientific question answering.

## 2. Objectives
- Create a Next.js frontend using JavaScript.
- Build a simple and user-friendly interface.
- Implement PDF upload functionality.
- Allow users to submit research questions.
- Display generated answers and source references.
- Connect the frontend to the FastAPI backend.

## 3. Tools and Technologies
- Next.js
- React
- JavaScript
- CSS
- Fetch API
- FastAPI (backend)

## 4. Input and Expected Output

**Input:**
- Scientific PDF documents.
- Natural-language research questions.

**Expected Output:**
- Upload confirmation.
- AI-generated answers.
- Source references with document names and page numbers.
- Query execution time.

## 5. Implementation Workflow
1. Inspect the existing frontend directory.
2. Initialize the Next.js application.
3. Configure the backend API connection.
4. Create the application interface.
5. Implement PDF upload.
6. Implement question answering.
7. Test frontend-backend integration.

## 6. Expected Outcome
A functional web interface that allows users to upload scientific PDFs and ask questions about their contents using the Advance AI Research Assistant.

In [3]:
from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

FRONTEND_DIR = PROJECT_ROOT / "frontend"

print("Project root:", PROJECT_ROOT)
print("Frontend directory:", FRONTEND_DIR)
print("Frontend directory exists:", FRONTEND_DIR.exists())

if FRONTEND_DIR.exists():
    for item in sorted(FRONTEND_DIR.iterdir()):
        print(item.name)

Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Frontend directory: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/frontend
Frontend directory exists: True
.env.local
.gitignore
.next
AGENTS.md
README.md
app
eslint.config.mjs
jsconfig.json
next.config.mjs
node_modules
package-lock.json
package.json
public


In [4]:
from pathlib import Path

print("Frontend directory contents:")

items = list(FRONTEND_DIR.iterdir())

if not items:
    print("The directory is empty.")
else:
    for item in items:
        print(item.name)

Frontend directory contents:
AGENTS.md
.env.local
package-lock.json
app
eslint.config.mjs
node_modules
next.config.mjs
jsconfig.json
README.md
public
.gitignore
package.json
.next


In [5]:
print("Frontend files:")

for item in sorted(FRONTEND_DIR.iterdir()):
    print(item.name)

Frontend files:
.env.local
.gitignore
.next
AGENTS.md
README.md
app
eslint.config.mjs
jsconfig.json
next.config.mjs
node_modules
package-lock.json
package.json
public


In [6]:
from pathlib import Path

APP_DIR = FRONTEND_DIR / "app"

print("Files in app directory:")
for item in sorted(APP_DIR.iterdir()):
    print(item.name)

Files in app directory:
.ipynb_checkpoints
favicon.ico
globals.css
layout.js
page.js
page.module.css


In [7]:
import json

package_json = FRONTEND_DIR / "package.json"

with package_json.open("r", encoding="utf-8") as file:
    package_data = json.load(file)

print(json.dumps(package_data, indent=2))

{
  "name": "frontend",
  "version": "0.1.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start",
    "lint": "eslint"
  },
  "dependencies": {
    "next": "16.4.6",
    "react": "19.3.0",
    "react-dom": "19.3.0"
  },
  "devDependencies": {
    "eslint": "^9",
    "eslint-config-next": "16.4.6"
  }
}


In [8]:
env_file = FRONTEND_DIR / ".env.local"

env_content = """NEXT_PUBLIC_API_URL=http://127.0.0.1:8000
"""

env_file.write_text(env_content, encoding="utf-8")

print("Created:", env_file)
print(env_file.read_text(encoding="utf-8"))

Created: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/frontend/.env.local
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000



In [9]:
from pathlib import Path

retriever_file = (
    PROJECT_ROOT
    / "src"
    / "retrieval"
    / "hybrid_retriever.py"
)

source = retriever_file.read_text(encoding="utf-8")

print("add_chunks defined in source:", "def add_chunks(" in source)

add_chunks defined in source: True


In [10]:
from pathlib import Path

retriever_file = (
    PROJECT_ROOT
    / "src"
    / "retrieval"
    / "hybrid_retriever.py"
)

lines = retriever_file.read_text(encoding="utf-8").splitlines()

for i in range(270, min(330, len(lines))):
    print(f"{i + 1:03}: {lines[i]}")

271:         )
272: 
273:         # Step 4: Cross-Encoder reranking
274:         final_results = self.rerank_results(
275:             query,
276:             fused_results,
277:             top_k=final_top_k
278:         )
279: 
280:         return final_results
281: 
282:     def add_chunks(self, new_chunks: list[dict]) -> int:
283:         """
284:         Add newly uploaded document chunks to the BM25 corpus.
285:     
286:         Returns:
287:             Number of newly added chunks.
288:         """
289:         if not new_chunks:
290:             return 0
291:     
292:         # Avoid adding chunks that already exist.
293:         existing_ids = {
294:             chunk["chunk_id"] for chunk in self.chunks
295:         }
296:     
297:         unique_chunks = [
298:             chunk for chunk in new_chunks
299:             if chunk["chunk_id"] not in existing_ids
300:         ]
301:     
302:         if not unique_chunks:
303:             return 0
304:     
305:         # Ex

In [11]:
import json
from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
UPLOADS_DIR = PROJECT_ROOT / "data" / "uploads"

# 1. Check the persistent chunk file.
with CHUNKS_FILE.open("r", encoding="utf-8") as file:
    chunks = [json.loads(line) for line in file if line.strip()]

print("JSONL chunk count:", len(chunks))

# 2. Identify uploaded documents.
documents = {}

for chunk in chunks:
    doc_id = chunk["document_id"]
    documents.setdefault(doc_id, {
        "filename": chunk["filename"],
        "chunks": 0
    })
    documents[doc_id]["chunks"] += 1

print("\nDocuments in JSONL:")
for doc_id, info in documents.items():
    print(
        f"Document: {info['filename']} | "
        f"ID: {doc_id} | "
        f"Chunks: {info['chunks']}"
    )

# 3. Check uploaded PDF files.
print("\nFiles in uploads directory:")

if UPLOADS_DIR.exists():
    for pdf in UPLOADS_DIR.glob("*.pdf"):
        print(pdf.name)
else:
    print("Uploads directory does not exist.")

JSONL chunk count: 241

Documents in JSONL:
Document: 41598_2019_Article_38798.pdf | ID: 41598_2019_Article_38798 | Chunks: 60
Document: NIPS-2017-attention-is-all-you-need-Paper.pdf | ID: NIPS-2017-attention-is-all-you-need-Paper | Chunks: 35
Document: s41598-019-52937-5.pdf | ID: s41598-019-52937-5 | Chunks: 61
Document: 41598_2019_Article_38798.pdf | ID: 342b966e18b34dea8dd5d4be493c35a7 | Chunks: 60
Document: TransferLearninginDeepLearningModels.pdf | ID: 8697255877364d7ba2a6e8b5662da302 | Chunks: 25

Files in uploads directory:
34d81edc11fb4164b0d261bca0201026_TransferLearninginDeepLearningModels.pdf
8697255877364d7ba2a6e8b5662da302_TransferLearninginDeepLearningModels.pdf
342b966e18b34dea8dd5d4be493c35a7_41598_2019_Article_38798.pdf
9a7a44fee0fe451187d56f8e319bcd8d_TransferLearninginDeepLearningModels.pdf


In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

# Add the project root to Python's module search path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("src directory exists:", (PROJECT_ROOT / "src").exists())

Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Project root exists: True
src directory exists: True


In [14]:
from src.embeddings.embedding_model import ChromaVectorStore

CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

vector_store = ChromaVectorStore(CHROMA_DIR)

print("ChromaDB directory:", CHROMA_DIR)
print("ChromaDB exists:", CHROMA_DIR.exists())
print("ChromaDB vector count:", vector_store.count())

ChromaDB directory: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/chroma_db
ChromaDB exists: True
ChromaDB vector count: 241


In [18]:
from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

FRONTEND_DIR = PROJECT_ROOT / "frontend"

# Search frontend source files for the tracking parameter.
for file_path in FRONTEND_DIR.rglob("*"):
    if (
        file_path.is_file()
        and file_path.suffix in [".js", ".jsx", ".css", ".json"]
        and "node_modules" not in file_path.parts
        and ".next" not in file_path.parts
    ):
        try:
            content = file_path.read_text(encoding="utf-8")
            if "Advance AI Research Assistant · Scientific PDF RAG System" in content:
                print(f"\nFound in: {file_path.relative_to(FRONTEND_DIR)}")
                for line_number, line in enumerate(content.splitlines(), 1):
                    if "utm_source" in line or "chatgpt.com" in line:
                        print(f"Line {line_number}: {line.strip()}")
        except (UnicodeDecodeError, OSError):
            pass


Found in: app/page.js

Found in: app/.ipynb_checkpoints/page-checkpoint.js


In [1]:

from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

FILES_TO_INSPECT = [
    "frontend/app/page.js",
    "frontend/app/page.module.css",
    "frontend/app/globals.css",
    "frontend/app/layout.js",
    "backend/app/main.py",
    "src/generation/llm_provider.py",
]

for relative_path in FILES_TO_INSPECT:
    file_path = PROJECT_ROOT / relative_path

    print("\n" + "=" * 85)
    print(f"FILE: {relative_path}")
    print("=" * 85)

    if file_path.exists():
        content = file_path.read_text(encoding="utf-8")
        print(content)
    else:
        print("FILE NOT FOUND")



FILE: frontend/app/page.js

"use client";

import { useState } from "react";
import styles from "./page.module.css";

const API_URL =
  process.env.NEXT_PUBLIC_API_URL || "http://127.0.0.1:8000";

export default function Home() {
  const [selectedFile, setSelectedFile] = useState(null);
  const [uploadStatus, setUploadStatus] = useState("");
  const [uploading, setUploading] = useState(false);

  const [question, setQuestion] = useState("");
  const [answer, setAnswer] = useState(null);
  const [asking, setAsking] = useState(false);
  const [error, setError] = useState("");

  // Upload a scientific PDF.
  async function handleUpload(event) {
    event.preventDefault();

    if (!selectedFile) {
      setUploadStatus("Please select a PDF file.");
      return;
    }

    setUploading(true);
    setUploadStatus("");
    setError("");

    try {
      const formData = new FormData();
      formData.append("file", selectedFile);

      const response = await fetch(
        `${API_URL}/do